In [2]:
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report
!pip install tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, InputLayer


ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)
ERROR: No matching distribution found for tensorflow


ModuleNotFoundError: No module named 'tensorflow'

In [3]:
# Load dataset
data = pd.read_csv("heart_failure.csv")
print(data.info())
print('Classes and number of values in the dataset:', Counter(data['death_event']))


<class 'pandas.DataFrame'>
RangeIndex: 299 entries, 0 to 298
Data columns (total 15 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                299 non-null    int64  
 1   age                       299 non-null    float64
 2   anaemia                   299 non-null    str    
 3   creatinine_phosphokinase  299 non-null    int64  
 4   diabetes                  299 non-null    str    
 5   ejection_fraction         299 non-null    int64  
 6   high_blood_pressure       299 non-null    str    
 7   platelets                 299 non-null    float64
 8   serum_creatinine          299 non-null    float64
 9   serum_sodium              299 non-null    int64  
 10  sex                       299 non-null    str    
 11  smoking                   299 non-null    str    
 12  time                      299 non-null    int64  
 13  DEATH_EVENT               299 non-null    int64  
 14  death_event          

In [4]:
# Target
data['death_event'] = data['death_event'].map({'yes':1, 'no':0}).astype(np.float32)
y = data['death_event']
# Features
X = data[['age','anaemia','creatinine_phosphokinase','diabetes',
          'ejection_fraction','high_blood_pressure','platelets',
          'serum_creatinine','serum_sodium','sex','smoking','time']]

# Train/test split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)


(239, 12) (60, 12) (239,) (60,)


In [5]:
# Numeric + categorical features
num_features = [
    'age','creatinine_phosphokinase','ejection_fraction',
    'platelets','serum_creatinine','serum_sodium','time'
]
cat_features = ['anaemia','diabetes','high_blood_pressure','sex','smoking']

# ColumnTransformer: scale numeric, one-hot encode categorical
ct = ColumnTransformer(
    transformers=[
        ('scale', StandardScaler(), num_features),
        ('onehot', OneHotEncoder(drop='if_binary'), cat_features)
    ],
    remainder='passthrough'
)


In [6]:
X_train = ct.fit_transform(X_train).astype(np.float32)
X_test = ct.transform(X_test).astype(np.float32)


In [7]:
# Build model
model = Sequential()
model.add(InputLayer(input_shape=(X_train.shape[1],)))
model.add(Dense(64, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
model.fit(X_train, Y_train, epochs=100, batch_size=16, verbose=1)


NameError: name 'Sequential' is not defined

In [8]:
# Evaluate
loss, acc = model.evaluate(X_test, Y_test, verbose=0)
print("Loss", loss, "Accuracy:", acc)

# Predictions: probabilities
y_probs = model.predict(X_test, verbose=0)

# Convert to binary class labels (0 or 1)
y_estimate = (y_probs > 0.5).astype("int32").ravel()

print(y_estimate[:10]) 

# True labels are already 0/1 in Y_test
y_true = Y_test.astype("int32").ravel()

# Classification report
from sklearn.metrics import classification_report
print(classification_report(y_true, y_estimate))

NameError: name 'model' is not defined